# Chapter 11: Probabilistic Graphical Models - Implementation

In [ ]:
import numpy as np
from collections import defaultdict
import random

print('Imports complete')

In [ ]:
class BayesianNetwork:
    def __init__(self):
        self.nodes = {}
        self.cpts = {}
        self.parents = {}
    
    def add_node(self, name, parents=None):
        self.nodes[name] = True
        self.parents[name] = parents if parents else []
    
    def set_cpt(self, name, cpt):
        self.cpts[name] = cpt
    
    def query(self, var, evidence=None):
        # Simple enumeration inference
        if evidence is None:
            evidence = {}
        
        # Get all variables
        all_vars = list(self.nodes.keys())
        hidden = [v for v in all_vars if v not in evidence and v != var]
        
        # Enumerate all values
        probs = {}
        for val in [True, False]:
            prob = self._enumerate_all(hidden, {**evidence, var: val})
            probs[val] = prob
        
        # Normalize
        total = sum(probs.values())
        return {k: v/total for k, v in probs.items()}
    
    def _enumerate_all(self, vars, evidence):
        if not vars:
            return self._prob_given_parents(evidence)
        
        var = vars[0]
        rest = vars[1:]
        
        prob_true = self._enumerate_all(rest, {**evidence, var: True})
        prob_false = self._enumerate_all(rest, {**evidence, var: False})
        
        return prob_true + prob_false
    
    def _prob_given_parents(self, evidence):
        prob = 1.0
        for var in evidence:
            parent_vals = tuple(evidence[p] for p in self.parents[var])
            cpt = self.cpts[var]
            var_val = evidence[var]
            prob *= cpt.get(parent_vals, {}).get(var_val, 0.5)
        return prob

print('Bayesian Network implemented')

In [ ]:
print('=== Example: Simple Medical Diagnosis ===')

# Create network
bn = BayesianNetwork()
bn.add_node('Disease')
bn.add_node('Symptom', parents=['Disease'])

# Set CPTs
bn.set_cpt('Disease', {
    (): {True: 0.01, False: 0.99}
})
bn.set_cpt('Symptom', {
    (True,): {True: 0.9, False: 0.1},
    (False,): {True: 0.1, False: 0.9}
})

# Query
result = bn.query('Disease', evidence={'Symptom': True})
print(f'P(Disease | Symptom) = {result[True]:.3f}')

In [ ]:
def gibbs_sampling(bn, query, evidence, num_samples=1000):
    # Initialize state
    state = evidence.copy()
    for var in bn.nodes:
        if var not in state:
            state[var] = random.choice([True, False])
    
    counts = defaultdict(int)
    
    for _ in range(num_samples):
        for var in bn.nodes:
            if var not in evidence:
                # Sample from conditional
                parent_vals = tuple(state[p] for p in bn.parents[var])
                cpt = bn.cpts[var].get(parent_vals, {True: 0.5, False: 0.5})
                state[var] = random.random() < cpt[True]
        
        counts[state[query]] += 1
    
    total = sum(counts.values())
    return {k: v/total for k, v in counts.items()}

print('Gibbs sampling implemented')

In [ ]:
print('=== Gibbs Sampling Test ===')
result_gibbs = gibbs_sampling(bn, 'Disease', {'Symptom': True}, num_samples=10000)
print(f'P(Disease | Symptom) [Gibbs] = {result_gibbs[True]:.3f}')

## Real-World: Medical Diagnosis

Building a complete diagnostic system with multiple diseases and symptoms.